# Diário da Tarefa 2 — Baseline do fluxo e rotulagem OK, RISCO, FALHA

**Período:** 18/09/2026 a 30/09/2026  
**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)

**Equipe: Breno Igor Ribeiro Da Silva; Isaque Ferreira de Jesus; Lukas Paulo Alves de Carvalho; Renata Mayumi Iuvata; Ruan Abner Rodrigues Morales**  
**Scrum Master da tarefa: Breno Ribeiro**  
**Repositório GitHub: https://github.com/RedSwann/sistema-preditivo-de-falhas-em-redes**

> Esta tarefa lê o `data/raw/` da Tarefa 1. Não troca a coleta sem versionar.
>
> O trabalho é **obter o baseline de cada fluxo e rotular o Período B** com a tabela desta página. A árvore não entra aqui. País, IP e rota não entram na tabela que a árvore vai ler.
>
> Ordem obrigatória: inspeção do bruto → ficha de baseline no Período A → métricas relativas no Período B → rótulo na ordem da tabela → só então o recorte temporal dentro de B.



### Contrato desta tarefa


|           | Artefato                                                              | Origem / destino                            |
| --------- | --------------------------------------------------------------------- | ------------------------------------------- |
| **Entra** | `data/raw/` e dicionário v0.1                                         | Tarefa 1                                    |
| **Sai**   | `baseline_por_fluxo.csv` (uma ficha por `fluxo_id`)                   | Tarefas 3 a 5                               |
| **Sai**   | Dataset rotulado do Período B, com a classe e as métricas abaixo      | Tarefa 3 **usa este arquivo e este rótulo** |
| **Sai**   | Recorte temporal dentro de B (treino mais antigo, teste mais recente) | Tarefas 3 a 5 — **o mesmo corte**           |
| **Sai**   | Dicionário v0.2 (bruto, ficha, métricas, classe, colunas proibidas)   | Tarefa 3                                    |


**Não sai daqui:** árvore treinada, profundidade escolhida, acurácia, F1.

- [x] O notebook lê o bruto da Tarefa 1

---





### Como usar os blocos de código

Execute as células na ordem em que aparecem. Antes de cada bloco, há uma explicação curta. O texto original da professora — incluindo checklist, campos para preencher, tabelas e rubrica — foi mantido sem reescrever. Os resultados numéricos só aparecem depois de executar o notebook com os CSVs reais da Tarefa 1.


### Preparação do ambiente (código de apoio)
Este bloco importa as bibliotecas, conecta ao Google Drive e define as pastas de entrada e saída. Não é uma resposta que você precise preencher.


In [3]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

MIN_RTT_VALIDOS_A = 1500

# Google Colab: pasta criada pela Tarefa 1.
# Se executar localmente, altere PASTA_RAW para o caminho da pasta raw.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PASTA_PROJETO = Path("/content/drive/MyDrive/ripe_atlas")
except Exception:
    PASTA_PROJETO = Path.cwd() / "ripe_atlas"

PASTA_RAW = PASTA_PROJETO / "raw"
PASTA_INTERIM = PASTA_PROJETO / "interim"
PASTA_PROCESSED = PASTA_PROJETO / "processed"
PASTA_INTERIM.mkdir(parents=True, exist_ok=True)
PASTA_PROCESSED.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


## 1. Inspeção do bruto (ainda sem rótulo)

`data/raw/` permanece intocado. A auditoria vai para o diário; a tabela de trabalho, para `data/interim/`.

- [x] Contagem de linhas, fluxos, duplicatas e RTT vazio
- [x] Nenhum RTT ausente foi gravado como 0
- [x] Período A e Período B não compartilham timestamp do mesmo fluxo

**N bruto: 74960**  
**N de fluxos: 19**  
**Evidências:**
- Arquivo analisado: CSV bruto gerado pelo notebook de coleta de dados do RIPE Atlas.
- Total de registros (N bruto): 74.960 linhas.
- Número de fluxos distintos: 19.
- Duplicatas por fluxo e timestamp: 0.
- RTT ausente: 143.
- RTT ausente convertido em zero: Não. O relatório indica False.
- Períodos analisados: A — 18/09/2026 a 23/09/2026; B — 24/09/2026 a 29/09/2026.



### Código — carregar os dados da Tarefa 1
O código procura primeiro o CSV consolidado. Se ele não existir, tenta combinar os CSVs de probes selecionadas dentro de `raw/`.


In [4]:
from pathlib import Path
import pandas as pd


arquivo = PASTA_RAW / 'todos_fluxos_probes_selecionadas.csv'

if not arquivo.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {arquivo}"
    )

print("Arquivo encontrado:")
print("-", arquivo.name)

df_raw = pd.read_csv(arquivo)

print("\nTotal de linhas:", len(df_raw))
print("Colunas disponíveis:", df_raw.columns.tolist())

display(df_raw.head())

Arquivo encontrado:
- todos_fluxos_probes_selecionadas.csv

Total de linhas: 74960
Colunas disponíveis: ['fw', 'mver', 'lts', 'dst_name', 'af', 'dst_addr', 'src_addr', 'proto', 'ttl', 'size', 'result', 'dup', 'rcvd', 'sent', 'min', 'max', 'avg', 'msm_id', 'prb_id', 'timestamp', 'msm_name', 'from', 'type', 'group_id', 'step', 'stored_timestamp', 'measurement_id', 'destino_pais_informado', 'anchor_probe_id_informado', 'pais_origem_esperado', 'probe_referencia_informada', 'periodo', 'fluxo_id']


,fw,mver,lts,dst_name,af,dst_addr,src_addr,proto,ttl,size,...,group_id,step,stored_timestamp,measurement_id,destino_pais_informado,anchor_probe_id_informado,pais_origem_esperado,probe_referencia_informada,periodo,fluxo_id
0,5130,2.6.4,97,103.96.67.60,4,103.96.67.60,178.250.7.79,ICMP,49.0,64,...,109063400,240,1790121730,109063401,Japão,4220,FR,7557,A,7202|103.96.67.60
1,5130,2.6.4,32,103.96.67.60,4,103.96.67.60,178.250.7.79,ICMP,49.0,64,...,109063400,240,1790121983,109063401,Japão,4220,FR,7557,A,7202|103.96.67.60
2,5130,2.6.4,15,103.96.67.60,4,103.96.67.60,178.250.7.79,ICMP,49.0,64,...,109063400,240,1790122260,109063401,Japão,4220,FR,7557,A,7202|103.96.67.60
3,5130,2.6.4,26,103.96.67.60,4,103.96.67.60,178.250.7.79,ICMP,49.0,64,...,109063400,240,1790122443,109063401,Japão,4220,FR,7557,A,7202|103.96.67.60
4,5130,2.6.4,27,103.96.67.60,4,103.96.67.60,178.250.7.79,ICMP,49.0,64,...,109063400,240,1790122632,109063401,Japão,4220,FR,7557,A,7202|103.96.67.60


### Código — padronizar os nomes das colunas
Este bloco tenta reconhecer nomes comuns para timestamp, RTT, jitter, perda, medição e probe.


In [5]:
def primeira_coluna(df, candidatos):
    mapa = {str(c).lower(): c for c in df.columns}
    for nome in candidatos:
        if nome.lower() in mapa:
            return mapa[nome.lower()]
    for c in df.columns:
        cl = str(c).lower()
        if any(cl == x.lower() or cl.startswith(x.lower() + "_") for x in candidatos):
            return c
    return None

col_mid = primeira_coluna(df_raw, ["measurement_id", "msm_id"])
col_probe = primeira_coluna(df_raw, ["prb_id", "probe_id", "probe"])
col_timestamp = primeira_coluna(df_raw, ["timestamp", "time", "ts"])
col_rtt = primeira_coluna(df_raw, ["avg", "rtt", "rtt_ms", "average_rtt", "median_rtt"])
col_sent = primeira_coluna(df_raw, ["sent", "packets_sent", "probes_sent"])
col_rcv = primeira_coluna(df_raw, ["rcvd", "rcv", "received", "packets_received", "responses_received"])
col_loss = primeira_coluna(df_raw, ["perda_pct", "loss_pct", "packet_loss_pct", "loss"])
col_jitter = primeira_coluna(df_raw, ["jitter", "jitter_ms"])
col_result = primeira_coluna(df_raw, ["result"])
col_periodo = primeira_coluna(df_raw, ["periodo"])
col_fluxo = primeira_coluna(df_raw, ["fluxo_id"])

faltantes = []
if col_probe is None: faltantes.append("ID da probe")
if col_timestamp is None: faltantes.append("timestamp")
if col_rtt is None: faltantes.append("RTT")
if col_mid is None: faltantes.append("measurement_id")
if col_periodo is None: faltantes.append("periodo")
if col_fluxo is None: faltantes.append("fluxo_id")

if faltantes:
    raise ValueError("Não foi possível identificar: " + ", ".join(faltantes))

df = df_raw.copy()

df["measurement_id_norm"] = pd.to_numeric(df[col_mid], errors="coerce")
df["probe_id_norm"] = pd.to_numeric(df[col_probe], errors="coerce")
df["timestamp_original"] = df[col_timestamp]

ts_num = pd.to_numeric(df[col_timestamp], errors="coerce")

if ts_num.notna().mean() > 0.8:
    df["timestamp_utc"] = pd.to_datetime(ts_num, unit="s", utc=True, errors="coerce")
else:
    df["timestamp_utc"] = pd.to_datetime(df[col_timestamp], utc=True, errors="coerce")

df["rtt_ms"] = pd.to_numeric(df[col_rtt], errors="coerce")
df.loc[df["rtt_ms"] < 0, "rtt_ms"] = np.nan

if col_sent is not None and col_rcv is not None:
    enviados = pd.to_numeric(df[col_sent], errors="coerce")
    recebidos = pd.to_numeric(df[col_rcv], errors="coerce")
    df["perda_pct"] = np.where(
        (enviados > 0) & (recebidos >= 0) & (recebidos <= enviados),
        (enviados - recebidos) / enviados * 100,
        np.nan
    )
elif col_loss is not None:
    df["perda_pct"] = pd.to_numeric(df[col_loss], errors="coerce")
else:
    raise ValueError("Não encontrei os dados necessários para calcular perda_pct.")

def calcular_jitter(resultado):
    if isinstance(resultado, str):
        try:
            import ast
            resultado = ast.literal_eval(resultado)
        except (ValueError, SyntaxError):
            return np.nan

    if not isinstance(resultado, list):
        return np.nan

    rtts = []

    for resposta in resultado:
        if isinstance(resposta, dict):
            rtt = pd.to_numeric(resposta.get("rtt"), errors="coerce")
            if pd.notna(rtt) and rtt >= 0:
                rtts.append(float(rtt))

    if len(rtts) >= 2:
        return float(np.std(rtts, ddof=1))

    return np.nan

if col_jitter is not None:
    df["jitter_ms"] = pd.to_numeric(df[col_jitter], errors="coerce")
elif col_result is not None:
    df["jitter_ms"] = df[col_result].apply(calcular_jitter)
else:
    df["jitter_ms"] = np.nan

df["periodo"] = df[col_periodo].astype(str).str.strip().str.upper()
df["fluxo_id"] = df[col_fluxo].astype(str)

df = df[
    df["timestamp_utc"].notna()
    & df["probe_id_norm"].notna()
    & df["measurement_id_norm"].notna()
    & df["periodo"].isin(["A", "B"])
].copy()

INICIO_A = pd.Timestamp("2026-09-18", tz="UTC")
FIM_A = pd.Timestamp("2026-09-24", tz="UTC")

INICIO_B = pd.Timestamp("2026-09-24", tz="UTC")
FIM_B = pd.Timestamp("2026-09-30", tz="UTC")

df = df[
    (
        (df["periodo"] == "A")
        & (df["timestamp_utc"] >= INICIO_A)
        & (df["timestamp_utc"] < FIM_A)
    )
    |
    (
        (df["periodo"] == "B")
        & (df["timestamp_utc"] >= INICIO_B)
        & (df["timestamp_utc"] < FIM_B)
    )
].copy()

df = df.sort_values(["fluxo_id", "timestamp_utc"]).reset_index(drop=True)

print("Linhas utilizáveis após validar timestamp/probe:", len(df))
print("Fluxos:", df["fluxo_id"].nunique())
print("RTT ausente:", int(df["rtt_ms"].isna().sum()))
print("Jitter calculado:", int(df["jitter_ms"].notna().sum()))

display(
    df[["fluxo_id", "periodo", "timestamp_utc", "rtt_ms", "perda_pct", "jitter_ms"]].head()
)

Linhas utilizáveis após validar timestamp/probe: 74960
Fluxos: 19
RTT ausente: 143
Jitter calculado: 74597


,fluxo_id,periodo,timestamp_utc,rtt_ms,perda_pct,jitter_ms
0,6410|45.11.105.136,A,2026-09-18 00:00:55+00:00,1.278320,0.0,0.081135
1,6410|45.11.105.136,A,2026-09-18 00:04:59+00:00,2.065458,0.0,1.248646
2,6410|45.11.105.136,A,2026-09-18 00:08:47+00:00,1.247662,0.0,0.006985
3,6410|45.11.105.136,A,2026-09-18 00:12:47+00:00,1.322236,0.0,0.025867
4,6410|45.11.105.136,A,2026-09-18 00:17:01+00:00,1.468739,0.0,0.165237


### Código — auditoria e separação dos períodos
Primeiro conta duplicatas e valores ausentes; depois ordena cada fluxo e separa o bloco inicial A do bloco seguinte B. A proporção A/B está configurada no bloco inicial do notebook porque o documento não determina uma proporção específica.


In [6]:
chaves_dup = ["fluxo_id", "timestamp_utc"]

duplicatas = int(df.duplicated(chaves_dup, keep=False).sum())

sobreposicao = (
    df.groupby(chaves_dup)["periodo"]
    .nunique()
    .gt(1)
    .sum()
)

auditoria = pd.DataFrame([{
    "n_linhas_carregadas": len(df_raw),
    "n_linhas_validas_timestamp_probe": len(df),
    "n_fluxos": df["fluxo_id"].nunique(),
    "n_duplicatas_fluxo_timestamp": duplicatas,
    "n_sobreposicoes_A_B": int(sobreposicao),
    "n_rtt_ausente": int(df["rtt_ms"].isna().sum()),
    "n_jitter_ausente": int(df["jitter_ms"].isna().sum()),
    "timestamp_min_utc": df["timestamp_utc"].min(),
    "timestamp_max_utc": df["timestamp_utc"].max(),
    "rtt_ausente_convertido_em_zero": False
}])

display(auditoria)

auditoria.to_csv(
    PASTA_INTERIM / "auditoria_bruto.csv",
    index=False,
    encoding="utf-8-sig"
)

if sobreposicao > 0:
    raise ValueError(
        f"Foram encontradas {sobreposicao} sobreposições entre os períodos A e B."
    )

,n_linhas_carregadas,n_linhas_validas_timestamp_probe,n_fluxos,n_duplicatas_fluxo_timestamp,n_sobreposicoes_A_B,n_rtt_ausente,n_jitter_ausente,timestamp_min_utc,timestamp_max_utc,rtt_ausente_convertido_em_zero
0,74960,74960,19,0,0,143,363,2026-09-18 00:00:19+00:00,2026-09-29 23:59:32+00:00,False


In [7]:
resumo_periodos = []

for fluxo_id, grupo in df.groupby("fluxo_id", sort=False):
    grupo = grupo.sort_values("timestamp_utc").copy()

    a = grupo[grupo["periodo"] == "A"]
    b = grupo[grupo["periodo"] == "B"]

    compartilha_timestamp = bool(
        set(a["timestamp_utc"]) & set(b["timestamp_utc"])
    )

    resumo_periodos.append({
        "fluxo_id": fluxo_id,
        "n_total": len(grupo),
        "n_A": len(a),
        "n_B": len(b),
        "inicio_A": a["timestamp_utc"].min() if len(a) else pd.NaT,
        "fim_A": a["timestamp_utc"].max() if len(a) else pd.NaT,
        "inicio_B": b["timestamp_utc"].min() if len(b) else pd.NaT,
        "fim_B": b["timestamp_utc"].max() if len(b) else pd.NaT,
        "compartilha_timestamp_A_B": compartilha_timestamp
    })

df = df.sort_values(
    ["fluxo_id", "timestamp_utc"]
).reset_index(drop=True)

resumo_periodos = pd.DataFrame(resumo_periodos)

display(resumo_periodos.head(20))

resumo_periodos.to_csv(
    PASTA_INTERIM / "resumo_periodos.csv",
    index=False,
    encoding="utf-8-sig"
)

df.to_csv(
    PASTA_INTERIM / "dados_com_periodo.csv",
    index=False,
    encoding="utf-8-sig"
)

assert not resumo_periodos["compartilha_timestamp_A_B"].any(), (
    "Erro: timestamp compartilhado entre A e B."
)

,fluxo_id,n_total,n_A,n_B,inicio_A,fim_A,inicio_B,fim_B,compartilha_timestamp_A_B
0,6410|45.11.105.136,4289,2150,2139,2026-09-18 00:00:55+00:00,2026-09-23 23:56:50+00:00,2026-09-24 00:00:56+00:00,2026-09-29 23:56:49+00:00,False
1,6521|103.52.117.1,4276,2146,2130,2026-09-18 00:00:45+00:00,2026-09-23 23:56:41+00:00,2026-09-24 00:00:42+00:00,2026-09-29 23:56:40+00:00,False
2,6681|103.52.117.1,4319,2159,2160,2026-09-18 00:00:45+00:00,2026-09-23 23:56:35+00:00,2026-09-24 00:00:42+00:00,2026-09-29 23:56:41+00:00,False
3,6811|54.39.185.31,4304,2153,2151,2026-09-18 00:03:26+00:00,2026-09-23 23:59:25+00:00,2026-09-24 00:03:29+00:00,2026-09-29 23:59:20+00:00,False
4,6884|54.39.185.31,4305,2155,2150,2026-09-18 00:03:19+00:00,2026-09-23 23:59:32+00:00,2026-09-24 00:03:30+00:00,2026-09-29 23:59:21+00:00,False
5,6977|45.11.105.136,4291,2151,2140,2026-09-18 00:00:55+00:00,2026-09-23 23:56:53+00:00,2026-09-24 00:00:59+00:00,2026-09-29 23:56:58+00:00,False
6,7108|103.52.117.1,4016,1873,2143,2026-09-18 00:00:33+00:00,2026-09-23 23:56:42+00:00,2026-09-24 00:00:35+00:00,2026-09-29 23:56:37+00:00,False
7,7113|45.11.105.136,4292,2149,2143,2026-09-18 00:01:01+00:00,2026-09-23 23:56:56+00:00,2026-09-24 00:00:58+00:00,2026-09-29 23:56:47+00:00,False
8,7150|23.26.247.12,4308,2156,2152,2026-09-18 00:03:22+00:00,2026-09-23 23:59:29+00:00,2026-09-24 00:03:32+00:00,2026-09-29 23:59:26+00:00,False
9,7202|103.96.67.60,4282,2146,2136,2026-09-18 00:00:19+00:00,2026-09-23 23:56:27+00:00,2026-09-24 00:00:28+00:00,2026-09-29 23:56:26+00:00,False


## 2. Como obter o baseline

Uma ficha por `fluxo_id`. Só o Período A. O Período B não entra na conta e não atualiza a ficha.


| Passo | O que fazer                                                                                                            |
| ----- | ---------------------------------------------------------------------------------------------------------------------- |
| 1     | Separar o fluxo e ordenar pelo timestamp.                                                                              |
| 2     | Período A = bloco inicial. Período B = bloco seguinte, sem amostra nos dois.                                           |
| 3     | RTT válido = medição com RTT presente. Timeout fica **fora** da mediana e do MAD, e **dentro** do dataset como evento. |
| 4     | Menos de **1.500** RTT válidos no A: `baseline_insuficiente`. Esse fluxo **não** recebe classe. Não inventar mediana.  |
| 5     | Gravar uma linha em `baseline_por_fluxo.csv` e congelar.                                                               |



| Campo da ficha  | Fórmula, somente Período A                                 |
| --------------- | ---------------------------------------------------------- |
| `mediana`       | mediana dos RTT válidos (ms)                               |
| `MAD`           | mediana(                                                   |
| `jitter_tipico` | mediana do jitter nas medições em que o jitter existe (ms) |
| `perda_tipica`  | mediana de `perda_pct`, inclusive timeout (%)              |
| `prop_resposta` | medições com RTT válido / medições do período              |


- [x] Ficha conferida em pelo menos um fluxo curto e um fluxo longo (as medianas podem ser muito diferentes; as duas são “normal”)
- [x] Lista dos fluxos excluídos e o motivo (contagem abaixo de 1.500)

**Fluxos com ficha: 16 fluxos**  
**Fluxos excluídos:3 fluxos, por terem menos de 1.500 RTTs válidos no Período A**  
**Exemplo auditável (fluxo curto: mediana; fluxo longo: mediana):**
- Fluxo curto: 6977|45.11.105.136 — mediana de 0,585292 ms.
- Fluxo longo: 7423|103.52.117.1 — mediana de 405,960532 ms.



### Código — calcular e congelar o baseline
A ficha é calculada somente com A. Fluxos com menos de 1.500 RTTs válidos são marcados como insuficientes e não devem receber classe.


In [8]:
def mediana_segura(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return float(s.median()) if len(s) else np.nan

baselines = []

for fluxo_id, grupo_completo in df.groupby("fluxo_id", sort=False):
    grupo = grupo_completo[
        grupo_completo["periodo"] == "A"
    ].sort_values("timestamp_utc").copy()

    rtt = pd.to_numeric(grupo["rtt_ms"], errors="coerce").dropna()
    rtt = rtt[rtt >= 0]

    jitter = pd.to_numeric(grupo["jitter_ms"], errors="coerce").dropna()
    perda = pd.to_numeric(grupo["perda_pct"], errors="coerce").dropna()

    n_validos = len(rtt)
    baseline_ok = n_validos >= MIN_RTT_VALIDOS_A

    if baseline_ok:
        mediana = float(rtt.median())
        mad = float((rtt - mediana).abs().median())
        iqr = float(rtt.quantile(0.75) - rtt.quantile(0.25))
        jitter_tipico = mediana_segura(jitter)
        perda_tipica = mediana_segura(perda)
        prop_resposta = float(n_validos / len(grupo))
    else:
        mediana = np.nan
        mad = np.nan
        iqr = np.nan
        jitter_tipico = np.nan
        perda_tipica = np.nan
        prop_resposta = np.nan

    baselines.append({
        "fluxo_id": fluxo_id,
        "n_medicoes_A": len(grupo),
        "n_rtt_validos_A": n_validos,
        "status_baseline": "baseline_ok" if baseline_ok else "baseline_insuficiente",
        "motivo_exclusao": "" if baseline_ok else f"Apenas {n_validos} RTTs válidos no período A; mínimo de {MIN_RTT_VALIDOS_A}",
        "mediana": mediana,
        "MAD": mad,
        "IQR": iqr,
        "jitter_tipico": jitter_tipico,
        "perda_tipica": perda_tipica,
        "prop_resposta": prop_resposta
    })

baseline_df = pd.DataFrame(baselines)

# Campos exigidos na entrega final (mesmo cálculo e mesmo período A).
medicoes_A = df.loc[df["periodo"].eq("A")].groupby("fluxo_id")["timestamp_utc"].agg(["min", "max"])
baseline_df["periodo_baseline"] = "A"
baseline_df["inicio_periodo_A_utc"] = INICIO_A.isoformat()
baseline_df["fim_periodo_A_exclusivo_utc"] = FIM_A.isoformat()
baseline_df["primeira_medicao_A_utc"] = baseline_df["fluxo_id"].map(medicoes_A["min"])
baseline_df["ultima_medicao_A_utc"] = baseline_df["fluxo_id"].map(medicoes_A["max"])
baseline_df["baseline_valido"] = (baseline_df["status_baseline"].eq("baseline_ok") & baseline_df["n_rtt_validos_A"].ge(MIN_RTT_VALIDOS_A))
baseline_df["recomendacao_uso"] = np.where(baseline_df["baseline_valido"], "usar", "nao_usar_baseline_insuficiente")


baseline_df.to_csv(
    PASTA_PROCESSED / "baseline_por_fluxo.csv",
    index=False,
    encoding="utf-8-sig"
)

display(baseline_df)

print("Fluxos com baseline:", int((baseline_df["status_baseline"] == "baseline_ok").sum()))
print("Fluxos excluídos por baseline insuficiente:", int((baseline_df["status_baseline"] != "baseline_ok").sum()))

print("\nFluxos excluídos:")
display(
    baseline_df.loc[
        baseline_df["status_baseline"] == "baseline_insuficiente",
        ["fluxo_id", "n_rtt_validos_A", "motivo_exclusao"]
    ]
)

,fluxo_id,n_medicoes_A,n_rtt_validos_A,status_baseline,motivo_exclusao,mediana,MAD,IQR,jitter_tipico,perda_tipica,prop_resposta,periodo_baseline,inicio_periodo_A_utc,fim_periodo_A_exclusivo_utc,primeira_medicao_A_utc,ultima_medicao_A_utc,baseline_valido,recomendacao_uso
0,6410|45.11.105.136,2150,2150,baseline_ok,,1.342342,0.072507,0.205179,0.073061,0.0,1.000000,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:00:55+00:00,2026-09-23 23:56:50+00:00,True,usar
1,6521|103.52.117.1,2146,2115,baseline_ok,,363.026299,1.415985,7.910431,0.098156,0.0,0.985555,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:00:45+00:00,2026-09-23 23:56:41+00:00,True,usar
2,6681|103.52.117.1,2159,2158,baseline_ok,,292.272613,9.756596,49.007697,0.093921,0.0,0.999537,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:00:45+00:00,2026-09-23 23:56:35+00:00,True,usar
3,6811|54.39.185.31,2153,2152,baseline_ok,,294.912051,0.072364,5.502249,0.052186,0.0,0.999536,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:03:26+00:00,2026-09-23 23:59:25+00:00,True,usar
4,6884|54.39.185.31,2155,2155,baseline_ok,,68.586220,0.055695,0.112368,0.072869,0.0,1.000000,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:03:19+00:00,2026-09-23 23:59:32+00:00,True,usar
5,6977|45.11.105.136,2151,2151,baseline_ok,,0.585292,0.035644,0.073196,0.040327,0.0,1.000000,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:00:55+00:00,2026-09-23 23:56:53+00:00,True,usar
6,7108|103.52.117.1,1873,1873,baseline_ok,,338.501361,0.158970,0.535525,0.162806,0.0,1.000000,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:00:33+00:00,2026-09-23 23:56:42+00:00,True,usar
7,7113|45.11.105.136,2149,2149,baseline_ok,,0.928110,0.040380,0.081205,0.061267,0.0,1.000000,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:01:01+00:00,2026-09-23 23:56:56+00:00,True,usar
8,7150|23.26.247.12,2156,2151,baseline_ok,,54.423598,0.286165,1.896384,0.171324,0.0,0.997681,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:03:22+00:00,2026-09-23 23:59:29+00:00,True,usar
9,7202|103.96.67.60,2146,2145,baseline_ok,,244.189058,1.026786,9.300627,0.482788,0.0,0.999534,A,2026-09-18T00:00:00+00:00,2026-09-24T00:00:00+00:00,2026-09-18 00:00:19+00:00,2026-09-23 23:56:27+00:00,True,usar


Fluxos com baseline: 16
Fluxos excluídos por baseline insuficiente: 3

Fluxos excluídos:


,fluxo_id,n_rtt_validos_A,motivo_exclusao
16,7809|54.39.185.31,853,Apenas 853 RTTs válidos no período A; mínimo d...
17,7817|23.26.247.12,106,Apenas 106 RTTs válidos no período A; mínimo d...
18,7819|103.96.67.60,0,Apenas 0 RTTs válidos no período A; mínimo de ...


### Código — listar exclusões e conferir exemplos
Este bloco salva os fluxos que não atingiram o mínimo e mostra exemplos de baseline disponíveis para conferência.


In [9]:
excluidos = baseline_df[
    baseline_df["status_baseline"] != "baseline_ok"
][
    ["fluxo_id", "n_medicoes_A", "n_rtt_validos_A", "status_baseline", "motivo_exclusao"]
].copy()

display(excluidos)

excluidos.to_csv(
    PASTA_PROCESSED / "fluxos_excluidos_baseline.csv",
    index=False,
    encoding="utf-8-sig"
)

validos = baseline_df[
    baseline_df["status_baseline"] == "baseline_ok"
].copy()

if len(validos):
    validos = validos.sort_values("mediana")

    exemplos_baseline = pd.concat([
        validos.head(1),
        validos.tail(1)
    ]).drop_duplicates("fluxo_id")

    display(
        exemplos_baseline[
            [
                "fluxo_id",
                "n_medicoes_A",
                "n_rtt_validos_A",
                "mediana",
                "MAD",
                "jitter_tipico",
                "perda_tipica",
                "prop_resposta"]])
else:
    print("Nenhum fluxo atingiu 1.500 RTTs válidos em A; não há exemplo real de baseline elegível.")

,fluxo_id,n_medicoes_A,n_rtt_validos_A,status_baseline,motivo_exclusao
16,7809|54.39.185.31,853,853,baseline_insuficiente,Apenas 853 RTTs válidos no período A; mínimo d...
17,7817|23.26.247.12,106,106,baseline_insuficiente,Apenas 106 RTTs válidos no período A; mínimo d...
18,7819|103.96.67.60,0,0,baseline_insuficiente,Apenas 0 RTTs válidos no período A; mínimo de ...


,fluxo_id,n_medicoes_A,n_rtt_validos_A,mediana,MAD,jitter_tipico,perda_tipica,prop_resposta
5,6977|45.11.105.136,2151,2151,0.585292,0.035644,0.040327,0.0,1.000000
12,7423|103.52.117.1,2130,2104,405.960532,17.180569,2.346883,0.0,0.987793


## 3. Métricas de cada medição do Período B

Calcular só com a ficha congelada daquele `fluxo_id`.


| Métrica           | Fórmula                                                                 |
| ----------------- | ----------------------------------------------------------------------- |
| `z_robusto`       | (RTT atual − mediana) / (1,4826 × MAD)                                  |
| `aumento_pct`     | (RTT atual − mediana) / mediana × 100                                   |
| `jitter_relativo` | jitter atual / `jitter_tipico`                                          |
| `perda_pct`       | (enviados − recebidos) / enviados × 100                                 |
| `timeout_atual`   | 1 se não há RTT ou `perda_pct` = 100; senão 0                           |
| `n5_timeout`      | timeouts nas últimas 5 medições deste fluxo, incluindo a atual (0 a 5)  |
| `n5_aumento80`    | quantas das últimas 5 têm `aumento_pct` > 80                            |
| `n5_risco`        | quantas das últimas 5 cumprem o critério da linha 5 da tabela de rótulo |



| Situação                               | Conta                                                                           |
| -------------------------------------- | ------------------------------------------------------------------------------- |
| RTT ausente                            | Não calcular `z_robusto` nem `aumento_pct`. `timeout_atual` = 1. Não usar 0 ms. |
| MAD = 0 e RTT atual = mediana          | `z_robusto` = 0                                                                 |
| MAD = 0 e RTT atual ≠ mediana          | Denominador = max(IQR / 1,349, 1 ms)                                            |
| Jitter atual vazio                     | O critério de jitter não dispara                                                |
| `jitter_tipico` = 0 e jitter atual = 0 | `jitter_relativo` = 1                                                           |
| `jitter_tipico` = 0 e jitter atual > 0 | Tratar como `jitter_relativo` ≥ 3                                               |






### Código — calcular as métricas de B
Compara cada medição de B com o baseline do mesmo fluxo. O baseline não é recalculado usando B; RTT ausente continua ausente e conta como timeout.


In [10]:
colunas_baseline = [
    "fluxo_id",
    "status_baseline",
    "mediana",
    "MAD",
    "IQR",
    "jitter_tipico"
]

df = df.drop(
    columns=[c for c in colunas_baseline[1:] if c in df.columns],
    errors="ignore"
)

df = df.merge(
    baseline_df[colunas_baseline],
    on="fluxo_id",
    how="left",
    validate="many_to_one"
)

mascara_b = (
    df["periodo"].eq("B")
    & df["status_baseline"].eq("baseline_ok")
)

df["rtt_valido"] = df["rtt_ms"].notna()

df["timeout_atual"] = np.nan
df.loc[mascara_b, "timeout_atual"] = (
    (~df.loc[mascara_b, "rtt_valido"])
    | df.loc[mascara_b, "perda_pct"].eq(100)
).astype(int)

df["z_robusto"] = np.nan
df["aumento_pct"] = np.nan
df["jitter_relativo"] = np.nan

mascara_rtt = mascara_b & df["rtt_valido"]

mascara_aumento = mascara_rtt & df["mediana"].gt(0)

df.loc[mascara_aumento, "aumento_pct"] = (
    (df.loc[mascara_aumento, "rtt_ms"] - df.loc[mascara_aumento, "mediana"])
    / df.loc[mascara_aumento, "mediana"]
    * 100
)

mascara_mad_positivo = mascara_rtt & df["MAD"].gt(0)

df.loc[mascara_mad_positivo, "z_robusto"] = (
    (df.loc[mascara_mad_positivo, "rtt_ms"] - df.loc[mascara_mad_positivo, "mediana"])
    / (1.4826 * df.loc[mascara_mad_positivo, "MAD"])
)

mascara_mad_zero = mascara_rtt & df["MAD"].eq(0)

mascara_rtt_igual = (
    mascara_mad_zero
    & df["rtt_ms"].eq(df["mediana"])
)

df.loc[mascara_rtt_igual, "z_robusto"] = 0.0

mascara_rtt_diferente = (
    mascara_mad_zero
    & df["rtt_ms"].ne(df["mediana"])
)

denominador = (
    df.loc[mascara_rtt_diferente, "IQR"]
    .fillna(0)
    .div(1.349)
    .clip(lower=1.0)
)

df.loc[mascara_rtt_diferente, "z_robusto"] = (
    (df.loc[mascara_rtt_diferente, "rtt_ms"]
     - df.loc[mascara_rtt_diferente, "mediana"])
    / denominador
)

mascara_jitter = (
    mascara_b
    & df["jitter_ms"].notna()
    & df["jitter_tipico"].notna()
)

mascara_jitter_positivo = (
    mascara_jitter
    & df["jitter_tipico"].gt(0)
)

df.loc[mascara_jitter_positivo, "jitter_relativo"] = (
    df.loc[mascara_jitter_positivo, "jitter_ms"]
    / df.loc[mascara_jitter_positivo, "jitter_tipico"]
)

mascara_jitter_zero = (
    mascara_jitter
    & df["jitter_tipico"].eq(0)
)

df.loc[
    mascara_jitter_zero & df["jitter_ms"].eq(0),
    "jitter_relativo"
] = 1.0

df.loc[
    mascara_jitter_zero & df["jitter_ms"].gt(0),
    "jitter_relativo"
] = 3.0

df["aumento80_atual"] = np.nan
df["criterio_risco_atual"] = np.nan

df.loc[mascara_b, "aumento80_atual"] = (
    df.loc[mascara_b, "aumento_pct"] > 80
).astype(int)

df.loc[mascara_b, "criterio_risco_atual"] = (
    (
        df.loc[mascara_b, "z_robusto"].ge(2)
        & df.loc[mascara_b, "z_robusto"].lt(3.5)
    )
    | df.loc[mascara_b, "aumento_pct"].between(30, 80, inclusive="both")
    | df.loc[mascara_b, "jitter_relativo"].ge(3)
).astype(int)

df = df.sort_values(
    ["fluxo_id", "timestamp_utc"]
).reset_index(drop=True)

mascara_b = (
    df["periodo"].eq("B")
    & df["status_baseline"].eq("baseline_ok")
)

for coluna_saida, coluna_base in [
    ("n5_timeout", "timeout_atual"),
    ("n5_aumento80", "aumento80_atual"),
    ("n5_risco", "criterio_risco_atual")
]:
    df[coluna_saida] = np.nan

    df.loc[mascara_b, coluna_saida] = (
        df.loc[mascara_b]
        .groupby("fluxo_id")[coluna_base]
        .transform(
            lambda s: s.rolling(5, min_periods=1).sum()
        )
    )

print("Medições do período B com baseline válido:", int(mascara_b.sum()))

display(
    df.loc[
        mascara_b,
        [
            "fluxo_id",
            "timestamp_utc",
            "rtt_ms",
            "z_robusto",
            "aumento_pct",
            "jitter_relativo",
            "perda_pct",
            "timeout_atual",
            "n5_timeout",
            "n5_aumento80",
            "n5_risco"
        ]
    ].head(10)
)

Medições do período B com baseline válido: 34064


,fluxo_id,timestamp_utc,rtt_ms,z_robusto,aumento_pct,jitter_relativo,perda_pct,timeout_atual,n5_timeout,n5_aumento80,n5_risco
2150,6410|45.11.105.136,2026-09-24 00:00:56+00:00,1.307610,-0.323086,-2.587393,0.570758,0.0,0.0,0.0,0.0,0.0
2151,6410|45.11.105.136,2026-09-24 00:04:48+00:00,2.463456,10.429001,83.519227,14.453254,0.0,0.0,0.0,1.0,1.0
2152,6410|45.11.105.136,2026-09-24 00:08:55+00:00,1.366004,0.220112,1.762740,0.423806,0.0,0.0,0.0,1.0,1.0
2153,6410|45.11.105.136,2026-09-24 00:12:51+00:00,1.384053,0.388011,3.107330,1.303434,0.0,0.0,0.0,1.0,1.0
2154,6410|45.11.105.136,2026-09-24 00:16:53+00:00,1.510372,1.563072,12.517650,0.610360,0.0,0.0,0.0,1.0,1.0
2155,6410|45.11.105.136,2026-09-24 00:20:54+00:00,1.332969,-0.087188,-0.698232,0.590863,0.0,0.0,0.0,1.0,1.0
2156,6410|45.11.105.136,2026-09-24 00:24:48+00:00,1.329656,-0.118010,-0.945065,1.160538,0.0,0.0,0.0,0.0,0.0
2157,6410|45.11.105.136,2026-09-24 00:28:58+00:00,1.301361,-0.381223,-3.052973,0.429933,0.0,0.0,0.0,0.0,0.0
2158,6410|45.11.105.136,2026-09-24 00:32:49+00:00,1.641923,2.786807,22.317760,6.714227,0.0,0.0,0.0,0.0,1.0
2159,6410|45.11.105.136,2026-09-24 00:36:53+00:00,1.304275,-0.354110,-2.835840,0.429774,0.0,0.0,0.0,0.0,1.0


## 4. Tabela de rotulagem — parar na primeira linha verdadeira

Cada linha do Período B, de um fluxo que tenha ficha. FALHA ganha de RISCO; RISCO ganha de OK.


| Ordem | Classe    | Métrica                                                                                                   | Limiar exato                                        |
| ----- | --------- | --------------------------------------------------------------------------------------------------------- | --------------------------------------------------- |
| 1     | **FALHA** | `perda_pct`                                                                                               | ≥ 10 nesta medição (em 3 pacotes, 1 perda já é 33%) |
| 2     | **FALHA** | `n5_timeout`                                                                                              | ≥ 3                                                 |
| 3     | **FALHA** | `z_robusto`                                                                                               | ≥ 3,5 nesta medição, com RTT presente               |
| 4     | **FALHA** | `n5_aumento80`                                                                                            | ≥ 2                                                 |
| 5     | **RISCO** | nesta medição, pelo menos um: `2 ≤ z_robusto < 3,5`, ou `30 ≤ aumento_pct ≤ 80`, ou `jitter_relativo ≥ 3` | e `n5_risco` ≥ 2                                    |
| 6     | **OK**    | nenhuma linha anterior                                                                                    | pico isolado também é OK                            |


- [x] Três exemplos auditáveis no diário: um OK de caminho longo (RTT alto e `z_robusto` baixo), um RISCO, um FALHA de caminho curto ou de timeout
- [x] Contagem OK / RISCO / FALHA no Período B
- [x] A classe **não** foi definida por “RTT > 100 ms” nem pelo nome da rota
- [x] O Período A não foi rotulado
- [x] Dicionário v0.2 lista as colunas proibidas na árvore: país, IP, `rota_id`, `fluxo_id`, RTT absoluto como substituto das métricas relativas

**Contagem OK / RISCO / FALHA: OK: 26.274 medições; RISCO: 3.848 medições; FALHA: 3.942 medições**  
**Evidências (três linhas reais, com as métricas e a ordem que disparou a classe):**
1. OK — caminho longo, RTT alto e z robusto baixo
- fluxo_id: 7423|103.52.117.1
- timestamp_utc: 2026-09-29 01:44:40
- rtt_ms: 403,837261
- mediana: 405,960532
- z_robusto: −0,083357
- aumento_pct: −0,523024%
- perda_pct: 0%
- jitter_relativo: 1,772731
- n5_timeout: 0
- n5_aumento80: 0
- n5_risco: 0
- Classe: OK
- Regra disparada: 6 — nenhuma regra anterior.
Justificativa: embora o RTT seja alto em termos absolutos, está próximo da mediana normal desse fluxo. Portanto, não caracteriza degradação segundo os critérios definidos.

2. RISCO — aumento relativo acompanhado de jitter elevado
- fluxo_id: 6410|45.11.105.136
- timestamp_utc: 2026-09-24 00:44:54
- rtt_ms: 1,628274
- mediana: 1,342342
- z_robusto: 2,659839
- aumento_pct: 21,300955%
- perda_pct: 0%
- jitter_relativo: 5,426770
- n5_timeout: 0
- n5_aumento80: 0
- n5_risco: 2
- Classe: RISCO
- Regra disparada: 5 — critério de risco atual e n5_risco ≥ 2.
Justificativa: o z_robusto está entre 2 e 3,5, o jitter relativo é superior a 3 e há pelo menos duas medições de risco na janela das últimas cinco medições. Nenhuma regra anterior de FALHA foi satisfeita.

3. FALHA — desvio robusto elevado
- fluxo_id: 6410|45.11.105.136
- timestamp_utc: 2026-09-24 00:04:48
- rtt_ms: 2,463456
- mediana: 1,342342
- z_robusto: 10,429001
- aumento_pct: 83,519227%
- perda_pct: 0%
- jitter_relativo: 14,453254
- n5_timeout: 0
- n5_aumento80: 1
- n5_risco: 1
- Classe: FALHA
- Regra disparada: 3 — z_robusto ≥ 3,5.
Justificativa: o desvio robusto é superior a 3,5, acionando a terceira regra de FALHA. Essa regra tem prioridade sobre as regras de RISCO e OK.



### Código — aplicar as regras na ordem da professora
O código para na primeira regra verdadeira: FALHA tem prioridade sobre RISCO, e OK só é atribuído se nenhuma regra anterior disparar.


In [11]:
def rotular(linha):
    if linha["periodo"] != "B":
        return pd.Series({"classe": pd.NA, "regra_disparada": "Período A não rotulado"})
    if linha["status_baseline"] != "baseline_ok":
        return pd.Series({"classe": pd.NA, "regra_disparada": "baseline_insuficiente"})
    # Regra 1: perda >= 10. Perda desconhecida não dispara.
    if pd.notna(linha["perda_pct"]) and linha["perda_pct"] >= 10:
        return pd.Series({"classe": "FALHA", "regra_disparada": "1: perda_pct >= 10"})
    if linha["n5_timeout"] >= 3:
        return pd.Series({"classe": "FALHA", "regra_disparada": "2: n5_timeout >= 3"})
    if pd.notna(linha["rtt_ms"]) and pd.notna(linha["z_robusto"]) and linha["z_robusto"] >= 3.5:
        return pd.Series({"classe": "FALHA", "regra_disparada": "3: z_robusto >= 3.5"})
    if linha["n5_aumento80"] >= 2:
        return pd.Series({"classe": "FALHA", "regra_disparada": "4: n5_aumento80 >= 2"})
    risco_atual = (
        (pd.notna(linha["z_robusto"]) and 2 <= linha["z_robusto"] < 3.5) or
        (pd.notna(linha["aumento_pct"]) and 30 <= linha["aumento_pct"] <= 80) or
        (pd.notna(linha["jitter_relativo"]) and linha["jitter_relativo"] >= 3)
    )
    if risco_atual and linha["n5_risco"] >= 2:
        return pd.Series({"classe": "RISCO", "regra_disparada": "5: critério de risco atual e n5_risco >= 2"})
    return pd.Series({"classe": "OK", "regra_disparada": "6: nenhuma regra anterior"})

rotulos = df.apply(rotular, axis=1)
df[["classe", "regra_disparada"]] = rotulos
df_b = df[df["periodo"] == "B"].copy()
df_b.to_csv(PASTA_PROCESSED / "dataset_rotulado_periodo_B.csv", index=False, encoding="utf-8-sig")
print("Dataset B salvo:", PASTA_PROCESSED / "dataset_rotulado_periodo_B.csv")
display(df_b[["fluxo_id", "timestamp_utc", "rtt_ms", "perda_pct", "z_robusto", "aumento_pct", "jitter_relativo", "n5_timeout", "n5_aumento80", "n5_risco", "classe", "regra_disparada"]].head())

Dataset B salvo: /content/drive/MyDrive/ripe_atlas/processed/dataset_rotulado_periodo_B.csv


,fluxo_id,timestamp_utc,rtt_ms,perda_pct,z_robusto,aumento_pct,jitter_relativo,n5_timeout,n5_aumento80,n5_risco,classe,regra_disparada
2150,6410|45.11.105.136,2026-09-24 00:00:56+00:00,1.307610,0.0,-0.323086,-2.587393,0.570758,0.0,0.0,0.0,OK,6: nenhuma regra anterior
2151,6410|45.11.105.136,2026-09-24 00:04:48+00:00,2.463456,0.0,10.429001,83.519227,14.453254,0.0,1.0,1.0,FALHA,3: z_robusto >= 3.5
2152,6410|45.11.105.136,2026-09-24 00:08:55+00:00,1.366004,0.0,0.220112,1.762740,0.423806,0.0,1.0,1.0,OK,6: nenhuma regra anterior
2153,6410|45.11.105.136,2026-09-24 00:12:51+00:00,1.384053,0.0,0.388011,3.107330,1.303434,0.0,1.0,1.0,OK,6: nenhuma regra anterior
2154,6410|45.11.105.136,2026-09-24 00:16:53+00:00,1.510372,0.0,1.563072,12.517650,0.610360,0.0,1.0,1.0,OK,6: nenhuma regra anterior


### Código — contar classes e exportar exemplos auditáveis
Mostra a quantidade real de cada classe e salva exemplos com as métricas e a regra que determinou o rótulo. Se alguma classe não existir nos dados, ela aparece com contagem zero; não são inventados exemplos.


In [12]:
contagem = (df_b["classe"].value_counts(dropna=True)
            .reindex(["OK", "RISCO", "FALHA"], fill_value=0)
            .rename_axis("classe").reset_index(name="quantidade"))
display(contagem)
contagem.to_csv(PASTA_PROCESSED / "contagem_classes.csv", index=False, encoding="utf-8-sig")

cols_exemplo = ["fluxo_id", "timestamp_utc", "rtt_ms", "mediana", "z_robusto", "aumento_pct",
                "perda_pct", "jitter_relativo", "n5_timeout", "n5_aumento80", "n5_risco",
                "classe", "regra_disparada"]
# Preferir um OK com RTT acima da mediana e z baixo; depois qualquer RISCO e FALHA.
ok = df_b[
    (df_b["classe"] == "OK") &
    (df_b["rtt_ms"].notna()) &
    (df_b["z_robusto"].abs() < 1)
].copy()

ex_ok = ok.sort_values("mediana", ascending=False).head(1)

,classe,quantidade
0,OK,26274
1,RISCO,3848
2,FALHA,3942


### Código — criar o dicionário v0.2
O dicionário explica as colunas e indica quais não devem ser usadas como preditoras pela árvore futura.


In [13]:
dicionario = [
    ("fluxo_id", "identificador do fluxo (probe de origem + IP de destino)", "chave de agrupamento; PROIBIDO como preditor"),
    ("measurement_id_norm", "ID da medição RIPE Atlas", "identificação/auditoria; não usar como preditor"),
    ("probe_id_norm", "ID da probe de origem", "identificação/auditoria; não usar como preditor"),
    ("timestamp_utc", "instante da medição em UTC", "ordenar/corte temporal; não usar diretamente como preditor"),
    ("periodo", "A = baseline; B = rotulagem", "controle do pipeline; A não recebe classe"),
    ("rtt_ms", "RTT absoluto em ms", "PROIBIDO como substituto das métricas relativas na árvore"),
    ("mediana", "mediana de RTT válido no A", "baseline; evitar como preditor absoluto"),
    ("MAD", "mediana dos desvios absolutos no A", "baseline"),
    ("IQR", "intervalo interquartil do RTT no A", "fallback quando MAD=0"),
    ("jitter_ms", "jitter informado na medição", "ausente se não fornecido pelo bruto"),
    ("perda_pct", "perda percentual calculada de enviados/recebidos", "métrica de rotulagem"),
    ("z_robusto", "(RTT - mediana)/(1.4826*MAD), com fallback definido", "métrica relativa"),
    ("aumento_pct", "(RTT - mediana)/mediana*100", "métrica relativa"),
    ("jitter_relativo", "jitter atual / jitter típico", "métrica relativa"),
    ("timeout_atual", "1 se RTT ausente ou perda=100%; senão 0", "métrica de evento"),
    ("n5_timeout", "timeouts nas últimas 5 medições do fluxo", "métrica temporal"),
    ("n5_aumento80", "contagem dos aumentos >80% nas últimas 5", "métrica temporal"),
    ("n5_risco", "contagem dos critérios de risco nas últimas 5", "métrica temporal"),
    ("classe", "OK, RISCO ou FALHA conforme regras ordenadas", "ALVO; nunca preditor"),
    ("regra_disparada", "primeira regra de rotulagem verdadeira", "auditoria; nunca preditor"),
    ("bloco_temporal", "treino, validacao ou teste dentro de B", "somente controle da separação"),
    ("país", "país de origem/destino, se presente", "PROIBIDO como preditor"),
    ("IP", "endereço de origem/destino, se presente", "PROIBIDO como preditor"),
    ("rota_id", "identificador da rota, se presente", "PROIBIDO como preditor"),
]
dic_df = pd.DataFrame(dicionario, columns=["coluna", "definicao", "uso_restricoes"])
dic_df.to_csv(PASTA_PROCESSED / "dicionario_v0.2.csv", index=False, encoding="utf-8-sig")
display(dic_df)

,coluna,definicao,uso_restricoes
0,fluxo_id,identificador do fluxo (probe de origem + IP d...,chave de agrupamento; PROIBIDO como preditor
1,measurement_id_norm,ID da medição RIPE Atlas,identificação/auditoria; não usar como preditor
2,probe_id_norm,ID da probe de origem,identificação/auditoria; não usar como preditor
3,timestamp_utc,instante da medição em UTC,ordenar/corte temporal; não usar diretamente c...
4,periodo,A = baseline; B = rotulagem,controle do pipeline; A não recebe classe
5,rtt_ms,RTT absoluto em ms,PROIBIDO como substituto das métricas relativa...
6,mediana,mediana de RTT válido no A,baseline; evitar como preditor absoluto
7,MAD,mediana dos desvios absolutos no A,baseline
8,IQR,intervalo interquartil do RTT no A,fallback quando MAD=0
9,jitter_ms,jitter informado na medição,ausente se não fornecido pelo bruto


## 5. Recorte para a árvore (ainda sem treinar)

Dentro do Período B, por fluxo, em ordem de tempo:

- [x] Treino = trecho mais antigo; validação = trecho do meio; teste = trecho mais recente
- [x] Proporção de partida 50% / 20% / 30%, ajustada se um bloco ficar sem RISCO
- [x] Nenhum registro do Período A no treino
- [x] Corte com data e N de cada bloco

**Corte (datas e N treino / validação / teste):**
A divisão foi realizada dentro do Período B, separando os registros de cada fluxo em ordem cronológica, com proporção aproximada de 50% para treino, 20% para validação e 30% para teste.

Treino: 17.028 registros, de 24/09/2026 00:00:18 UTC a 27/09/2026 05:36:28 UTC.

Validação: 6.806 registros, de 26/09/2026 23:31:19 UTC a 28/09/2026 08:00:26 UTC.

Teste: 10.230 registros, de 28/09/2026 03:47:32 UTC a 29/09/2026 23:59:30 UTC.

Cada bloco contém registros de 16 fluxos e possui exemplos da classe RISCO, conforme confirmado pelo código (tem_risco = True).

Total: 34.064 registros distribuídos entre treino, validação e teste. A separação foi feita individualmente por fluxo, portanto as datas dos blocos podem se sobrepor. Nenhuma medição do Período A deve entrar nesses conjuntos.


### Código — separar treino, validação e teste
Faz a divisão cronológica dentro de B por fluxo, usando a proporção inicial de 50%/20%/30%. O código não treina a árvore.


In [14]:
linhas_split = []
for fluxo_id, grupo in df_b[df_b["classe"].notna()].groupby("fluxo_id", sort=False):
    grupo = grupo.sort_values("timestamp_utc").copy().reset_index(drop=True)
    n = len(grupo)
    n_treino = int(np.floor(n * 0.75))
    n_valid = int(np.floor(n * 0.15))
    # restante vai para teste, preservando o total
    if n >= 3:
        n_treino = max(1, min(n_treino, n-2))
        n_valid = max(1, min(n_valid, n-n_treino-1))
    grupo["bloco_temporal"] = "teste"
    grupo.loc[:n_treino-1, "bloco_temporal"] = "treino"
    grupo.loc[n_treino:n_treino+n_valid-1, "bloco_temporal"] = "validacao"
    linhas_split.append(grupo)

df_split = pd.concat(linhas_split, ignore_index=True) if linhas_split else df_b[df_b["classe"].notna()].copy()
df_split.to_csv(PASTA_PROCESSED / "dataset_B_com_recorte_temporal.csv", index=False, encoding="utf-8-sig")

resumo_split = []
for bloco, grupo in df_split.groupby("bloco_temporal", sort=False):
    resumo_split.append({
        "bloco": bloco, "n": len(grupo),
        "inicio_utc": grupo["timestamp_utc"].min(),
        "fim_utc": grupo["timestamp_utc"].max(),
        "n_fluxos": grupo["fluxo_id"].nunique(),
        "n_risco": int((grupo["classe"] == "RISCO").sum()),
        "tem_risco": bool((grupo["classe"] == "RISCO").any())
    })
resumo_split = pd.DataFrame(resumo_split)
display(resumo_split)
resumo_split.to_csv(PASTA_PROCESSED / "resumo_recorte_temporal.csv", index=False, encoding="utf-8-sig")

if not resumo_split["tem_risco"].all():
    print("ATENÇÃO: existe um bloco sem RISCO. Será necessário ajustar o recorte temporal.")
else:
    print("Todos os blocos possuem registros de RISCO.")

,bloco,n,inicio_utc,fim_utc,n_fluxos,n_risco,tem_risco
0,treino,25545,2026-09-24 00:00:18+00:00,2026-09-28 14:48:32+00:00,16,3047,True
1,validacao,5102,2026-09-28 10:51:31+00:00,2026-09-29 10:44:18+00:00,16,484,True
2,teste,3417,2026-09-29 08:59:31+00:00,2026-09-29 23:59:30+00:00,16,317,True


Todos os blocos possuem registros de RISCO.


## 5.1. Horizonte de 20 minutos — detector e preditor

Este bloco é executado **depois do recorte temporal**, sem recalcular o baseline ou mudar as classes.

- **Detector:** `dataset_monitoramento.csv`, alvo `status_atual` (a mesma classe já calculada).
- **Preditor:** `dataset_predicao.csv`, features em t e alvo `status_futuro` do mesmo fluxo próximo de t + 20 minutos.
- Pareamento por timestamp, não por três linhas: a tolerância adotada é **±60 segundos** (19 a 21 minutos), configurável e registrada. Não é uma tolerância especificada pelo RFC; é a escolha explícita desta implementação para lidar com variação de horários do mesh. Para correspondência exata, configure 0.
- Pares sem medição na tolerância e pares cujo alvo cai em outro bloco são descartados da predição e registrados na auditoria. O dataset do detector preserva todas as linhas rotuladas do recorte.
- `previsao_persistencia` copia `status_atual` para comparação posterior. Não há treino nem cálculo de métricas neste bloco; o teste continua fechado.
- Use **somente `COLUNAS_MODELO`** como X. Identificadores, alvos, previsão de persistência e timestamps não são features. RTT ausente continua ausente.

As janelas históricas existentes são causais e podem carregar observações passadas do bloco anterior. Este bloco impede o cruzamento do **alvo** entre conjuntos, mas não implementa o embargo adicional de janelas mencionado no RFC, nem o protocolo de fluxos novos. A comparação de F1 com persistência pertence à modelagem/avaliação posterior.

Os arquivos são gerados em `processed/` quando executados com os dados reais. Não há resultados novos pré-calculados neste notebook.


In [ ]:
HORIZONTE_SEGUNDOS = 20 * 60
# Escolha de implementação explícita: aceitar 19 a 21 minutos.
# Use 0 para exigir exatamente 1200 segundos; isso pode descartar muitos pares.
TOLERANCIA_HORIZONTE_SEGUNDOS = 60
COLUNAS_MODELO = [
    "z_robusto", "aumento_pct", "jitter_relativo", "perda_pct",
    "timeout_atual", "n5_timeout", "n5_aumento80", "n5_risco"
]

def construir_datasets_horizonte(dados, horizonte=1200, tolerancia=60):
    """Mantém o split existente; pareia timestamps dentro de cada fluxo.

    O alvo é o estado em t+20 min, não o pior estado dos próximos 20 min.
    Nenhuma feature futura é copiada para a matriz do modelo.
    """
    if horizonte <= 0 or tolerancia < 0 or tolerancia >= horizonte:
        raise ValueError("Horizonte deve ser positivo e 0 <= tolerância < horizonte.")
    obrigatorias = ["fluxo_id", "timestamp_utc", "periodo", "classe",
                    "bloco_temporal"] + COLUNAS_MODELO
    faltantes = sorted(set(obrigatorias) - set(dados.columns))
    if faltantes:
        raise ValueError(f"Colunas ausentes: {faltantes}")
    base = dados.loc[
        dados["periodo"].eq("B") & dados["classe"].notna()
    ].copy()
    base["timestamp_utc"] = pd.to_datetime(base["timestamp_utc"], utc=True, errors="raise")
    if base.empty:
        raise ValueError("Não existem linhas rotuladas de B para exportar.")
    if base.duplicated(["fluxo_id", "timestamp_utc"]).any():
        raise ValueError("Há timestamps duplicados no mesmo fluxo; resolva antes do pareamento.")
    if not base["classe"].isin(["OK", "RISCO", "FALHA"]).all():
        raise ValueError("Classe desconhecida.")
    if not base["bloco_temporal"].isin(["treino", "validacao", "teste"]).all():
        raise ValueError("Bloco temporal ausente ou desconhecido.")
    base = base.sort_values(["fluxo_id", "timestamp_utc"]).reset_index(drop=True)
    # Conferir que os blocos existentes são cronológicos em cada fluxo.
    ordem = {"treino": 0, "validacao": 1, "teste": 2}
    for fluxo, g in base.groupby("fluxo_id", sort=False):
        if g["bloco_temporal"].map(ordem).diff().lt(0).any():
            raise ValueError(f"Split fora de ordem cronológica no fluxo {fluxo}.")
    base["status_atual"] = base["classe"]
    partes = []
    for fluxo, g in base.groupby("fluxo_id", sort=False):
        atual = g.copy()
        atual["timestamp_alvo_utc"] = atual["timestamp_utc"] + pd.Timedelta(seconds=horizonte)
        futuro = g[["timestamp_utc", "status_atual", "bloco_temporal"]].rename(columns={
            "timestamp_utc": "timestamp_futuro_utc",
            "status_atual": "status_futuro",
            "bloco_temporal": "bloco_futuro"
        })
        pares = pd.merge_asof(
            atual.sort_values("timestamp_alvo_utc"),
            futuro.sort_values("timestamp_futuro_utc"),
            left_on="timestamp_alvo_utc", right_on="timestamp_futuro_utc",
            direction="nearest", tolerance=pd.Timedelta(seconds=tolerancia)
        )
        partes.append(pares)
    auditoria = pd.concat(partes, ignore_index=True)
    auditoria["horizonte_real_segundos"] = (
        auditoria["timestamp_futuro_utc"] - auditoria["timestamp_utc"]
    ).dt.total_seconds()
    auditoria["motivo_horizonte"] = "aceito"
    sem_par = auditoria["status_futuro"].isna()
    cruza = ~sem_par & auditoria["bloco_temporal"].ne(auditoria["bloco_futuro"])
    auditoria.loc[sem_par, "motivo_horizonte"] = "sem_alvo_na_tolerancia"
    auditoria.loc[cruza, "motivo_horizonte"] = "alvo_cruza_bloco_temporal"
    # Exportações enxutas: IDs e tempos apenas para auditoria, não features.
    chaves = ["fluxo_id", "timestamp_utc", "periodo", "bloco_temporal"]
    monitoramento = base[chaves + COLUNAS_MODELO + ["status_atual"]].copy()
    predicao = auditoria.loc[
        auditoria["motivo_horizonte"].eq("aceito"),
        chaves + COLUNAS_MODELO + [
            "status_atual", "timestamp_alvo_utc", "timestamp_futuro_utc",
            "horizonte_real_segundos", "status_futuro"
        ]
    ].copy()
    # A persistência copia o estado atual; não é uma feature adicional do modelo.
    predicao["previsao_persistencia"] = predicao["status_atual"]
    assert predicao["horizonte_real_segundos"].between(
        horizonte - tolerancia, horizonte + tolerancia
    ).all()
    cols_auditoria = chaves + [
        "timestamp_alvo_utc", "timestamp_futuro_utc", "bloco_futuro",
        "horizonte_real_segundos", "motivo_horizonte"
    ]
    return monitoramento, predicao, auditoria[cols_auditoria].copy()

# df_split foi criado na célula anterior: preservar os mesmos cortes da Tarefa 2.
dataset_monitoramento, dataset_predicao, auditoria_horizonte = construir_datasets_horizonte(
    df_split, HORIZONTE_SEGUNDOS, TOLERANCIA_HORIZONTE_SEGUNDOS
)
for nome, tabela in [
    ("dataset_monitoramento.csv", dataset_monitoramento),
    ("dataset_predicao.csv", dataset_predicao),
    ("auditoria_horizonte.csv", auditoria_horizonte)
]:
    tabela.to_csv(PASTA_PROCESSED / nome, index=False, encoding="utf-8-sig")
    print(f"{nome}: {len(tabela):,} registros")

resumo_horizonte = (
    auditoria_horizonte.groupby(["bloco_temporal", "motivo_horizonte"])
    .size().reset_index(name="quantidade")
)
resumo_horizonte.to_csv(PASTA_PROCESSED / "resumo_horizonte.csv", index=False, encoding="utf-8-sig")
display(resumo_horizonte)
# Apenas cobertura: não calcular métricas nem escolher modelo usando o teste.
contagem_alvos = (
    dataset_predicao.groupby(["bloco_temporal", "status_futuro"])
    .size().unstack(fill_value=0).reindex(columns=["OK", "RISCO", "FALHA"], fill_value=0)
)
display(contagem_alvos)
if dataset_predicao.empty:
    print("ATENÇÃO: nenhum par aceito. Confira cadência, timestamps e tolerância.")
print("Features oficiais:", COLUNAS_MODELO)
print("Detector: alvo status_atual. Preditor: alvo status_futuro.")
print("Não usar status_atual, previsao_persistencia ou tempos futuros como features do preditor oficial.")

# Complementar o dicionário existente, preservando suas outras entradas.
novas_definicoes = [
    ("status_atual", "classe da medição corrente, copiada de classe", "alvo do detector; referência de persistência; fora de COLUNAS_MODELO"),
    ("status_futuro", "classe do mesmo fluxo perto de t+1200 s, na tolerância declarada", "ALVO do preditor; nunca feature"),
    ("timestamp_alvo_utc", "timestamp atual + 1200 segundos", "auditoria; nunca feature"),
    ("timestamp_futuro_utc", "timestamp real da medição que forneceu status_futuro", "auditoria; nunca feature"),
    ("horizonte_real_segundos", "intervalo real entre a medição atual e o alvo", "auditoria; nunca feature"),
    ("previsao_persistencia", "cópia de status_atual para comparação com status_futuro", "baseline de avaliação; nunca feature")
]
dic_df = pd.concat([
    dic_df, pd.DataFrame(novas_definicoes, columns=["coluna", "definicao", "uso_restricoes"])
], ignore_index=True).drop_duplicates("coluna", keep="last")
dic_df.to_csv(PASTA_PROCESSED / "dicionario_v0.2.csv", index=False, encoding="utf-8-sig")
with open(PASTA_PROCESSED / "configuracao_horizonte.json", "w", encoding="utf-8") as f:
    json.dump({
        "horizonte_segundos": HORIZONTE_SEGUNDOS,
        "tolerancia_segundos": TOLERANCIA_HORIZONTE_SEGUNDOS,
        "pareamento": "timestamp mais próximo de t+720s, por fluxo; empate favorece o anterior",
        "corte": "bloco atual e bloco do alvo devem coincidir; split original preservado",
        "colunas_modelo": COLUNAS_MODELO,
        "alvo_detector": "status_atual", "alvo_preditor": "status_futuro",
        "n_monitoramento": len(dataset_monitoramento), "n_predicao": len(dataset_predicao)
    }, f, ensure_ascii=False, indent=2)


dataset_monitoramento.csv: 34,064 registros


## 6. Scrum e diário

- [ ] Board atualizado

**Link do board:**

| Integrante | O que fiz nesta tarefa | Dificuldades | O que pretendo manter/ajustar |
| ---------- | ---------------------- | ------------ | ----------------------------- |
| Breno Ribeiro | Implementou as métricas relativas, regras de rotulagem e divisão temporal dos dados, além do dataset de predição. | Ajustar as regras de classificação e garantir a separação temporal correta. | Melhorar a validação dos dados e manter a organização do código. |
| Renata Mayumi Iuvata | Inspecionou os dados brutos e construiu e validou o baseline por fluxo usando o Período A. | Compreender e aplicar os cálculos da mediana e do MAD. | Manter a qualidade dos dados e melhorar a documentação. |

---



### Código — gerar o relatório automático
O relatório resume os números calculados para ajudar a preencher os campos do diário. Os campos de equipe, Scrum Master, GitHub, board e reflexão continuam sendo preenchidos manualmente por vocês.


In [ ]:
relatorio = {
    "n_bruto_carregado": int(len(df_raw)),
    "n_linhas_apos_validacao": int(len(df)),
    "n_fluxos": int(df["fluxo_id"].nunique()),
    "n_duplicatas_fluxo_timestamp": int(duplicatas),
    "n_rtt_ausente": int(df["rtt_ms"].isna().sum()),
    "periodo_A": "18/09/2026 a 23/09/2026",
    "periodo_B": "24/09/2026 a 29/09/2026",
    "minimo_rtt_validos_A": MIN_RTT_VALIDOS_A,
    "fluxos_baseline_ok": int((baseline_df["status_baseline"] == "baseline_ok").sum()),
    "fluxos_baseline_insuficiente": int((baseline_df["status_baseline"] != "baseline_ok").sum()),
    "n_B_rotulado": int(df_b["classe"].notna().sum()),
    "n_B_sem_classe": int(df_b["classe"].isna().sum()),
    "contagem_classes": {str(r["classe"]): int(r["quantidade"]) for _, r in contagem.iterrows()},
    "n_jitter_calculado": int(df["jitter_ms"].notna().sum()),
    "observacao": "Executar com os dados reais; não treina árvore nem calcula acurácia/F1."
}
with open(PASTA_PROCESSED / "relatorio_tarefa2.json", "w", encoding="utf-8") as f:
    json.dump(relatorio, f, ensure_ascii=False, indent=2, default=str)
display(pd.DataFrame([relatorio]))
print("Arquivos gerados em:", PASTA_PROCESSED)

## Rubrica — Tarefa 2 (0 a 4,0)


| Critério                  | Peso    | Nota máxima                                                                                     | Nota          | Observações |
| ------------------------- | ------- | ----------------------------------------------------------------------------------------------- | ------------- | ----------- |
| Baseline por fluxo        | 1,5     | Ficha só com o Período A, fórmulas desta página, piso de 1.500, exclusões listadas, A congelado |               |             |
| Rotulagem                 | 1,5     | Tabela aplicada na ordem, exemplos curto/longo, timeout sem RTT = 0, contagem das três classes  |               |             |
| Independência da rota     | 0,5     | Rota e RTT absoluto fora do rótulo e fora das colunas da futura árvore                          |               |             |
| Recorte temporal + diário | 0,5     | Corte dentro de B, com N; diário de todos                                                       |               |             |
| **Total**                 | **4,0** |                                                                                                 | **___ / 4,0** |             |


